# Segment Anything (MobileSAM) — pure C++ on GPU (Colab)

Runs the dependency-free C++ port of **MobileSAM** (TinyViT encoder + prompt encoder + mask decoder)
on a Colab GPU. No PyTorch at run time: the same headers, compiled with `nvcc -DUSE_CUDA` so every
matmul / conv GEMM offloads to **cuBLAS** via the `bk::gemm_hosted` seam. Builds CPU + GPU and times.

Runtime → Change runtime type → **GPU** (T4 is fine).


## 1. GPU + repo


In [ ]:
!nvidia-smi -L
!nvcc --version | tail -2


In [ ]:
%cd /content
![ -d segment_anything_cpp] || git clone https://github.com/yomei-o/segment_anything_cpp.git
%cd /content/segment_anything_cpp


## 2. Weights (one-time extraction from mobile_sam.pt — the only Python step)


In [ ]:
!pip -q install git+https://github.com/ChaoningZhang/MobileSAM.git timm numpy
import os, urllib.request
if not os.path.exists('pure/ref/mobile_sam.pt'):
    urllib.request.urlretrieve('https://github.com/ChaoningZhang/MobileSAM/raw/master/weights/mobile_sam.pt','pure/ref/mobile_sam.pt')
!cd pure/ref && python export_sam.py && python export_tinyvit.py


## 3. Build — CPU (g++) and GPU (cuBLAS via nvcc -DUSE_CUDA)


In [ ]:
!g++ -O2 -std=c++17 -DNOMINMAX -Ipure/third_party pure/infer_sam.cpp -o infer_cpu
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -diag-suppress 550 -Ipure/third_party pure/infer_sam.cpp -lcublas -o infer_gpu
print('built infer_cpu / infer_gpu')


## 4. Segment — click a point, overlay the mask (CPU vs GPU timing)
The encoder dominates (windowed attention over ~1072 windows); the GPU seam offloads its matmuls/convs.


In [ ]:
import urllib.request
urllib.request.urlretrieve('https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg','test.jpg')
import time
for n in ['infer_cpu','infer_gpu']:
    t=time.time(); os.system(f'./{n} test.jpg 320 240 out_{n}.png pure/ref'); print(f'{n:10s} {time.time()-t:5.1f}s')


In [ ]:
from IPython.display import Image, display
print('click (320,240) -> segmented:'); display(Image('test.jpg', width=360), Image('out_infer_gpu.png', width=360))


## 5. Train the mask decoder on GPU (synthetic)


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -diag-suppress 550 -Ipure/third_party pure/train_sam.cpp -lcublas -o train_gpu
!./train_gpu pure/ref --steps 20


## 6. Browser demo
`wasm/` is a click-to-segment demo (pure C++ -> WebAssembly). Enable GitHub Pages and open
`https://yomei-o.github.io/segment_anything_cpp/wasm/`.
